# 01 · Pipeline: de RAW a MASTER

**Caso transversal:** Online Retail II. *Dado un cliente y una fecha de corte, ¿volverá a comprar en los 90 días siguientes?*

Este cuaderno **es el pipeline**: lleva toda la lógica y se puede ejecutar de punta a punta (`Ejecutar todo`) o desde la terminal con
`papermill`. El paquete `ia_mlops` es su traducción a `.py` (ver `docs/de-notebook-a-python.md`).

Las capas se construyen en este orden: **RAW → MASTER → FS → ML**.

| Capa | Regla |
|---|---|
| **RAW** | El dato original, sin tocar, con linaje: de qué fila viene cada registro. |
| **MASTER** | El dato depurado y relacional. Se puede borrar y regenerar desde RAW. Lo que se descarta deja rastro. |

Antes de empezar, en VS Code elige el kernel **env-ia-mlops**.

In [ ]:
# Parámetros. papermill puede sobrescribirlos:  papermill 01_pipeline_raw_a_master.ipynb salida.ipynb -p FUENTE completo
# FUENTE: 'muestra' (CSV del repositorio, segundos) o 'completo' (Excel original de UCI, 1,07 millones de filas)
FUENTE = 'muestra'
# RUTA_DB: None usa data/processed/curso.db, o la variable de entorno CURSO_DB
RUTA_DB = None

In [ ]:
import os, sqlite3, sys
from pathlib import Path

import pandas as pd

pd.set_option('display.width', 150, 'display.max_columns', 30)

# Carpeta del proyecto: la que contiene «sql» y «pyproject.toml».
RAIZ = Path.cwd().resolve()
while not ((RAIZ / 'sql').exists() and (RAIZ / 'pyproject.toml').exists()) and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
SQL = RAIZ / 'sql'
DB = Path(RUTA_DB or os.environ.get('CURSO_DB') or RAIZ / 'data' / 'processed' / 'curso.db')
DB.parent.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')


def ejecutar_script(archivo):
    """Ejecuta un archivo .sql de la carpeta sql/ (puede tener varias sentencias)."""
    con.executescript((SQL / archivo).read_text(encoding='utf-8'))
    con.commit()


def escalar(consulta, parametros=None):
    fila = con.execute(consulta, parametros or {}).fetchone()
    return None if fila is None else fila[0]


print('Proyecto:', RAIZ)
print('Base de datos:', DB)

## 1. RAW: cargar el original sin limpiar nada
Se guarda **cada fila tal como viene**, más el linaje: `_id`, `_hoja` y `_fila`. `sql/raw/01_crear_raw.sql` define las tablas.

In [ ]:
import hashlib, math
from datetime import datetime

FUENTES = {
    'muestra': RAIZ / 'data' / 'raw' / 'muestra_online_retail_ii.csv',
    'completo': RAIZ / 'data' / 'raw' / 'online_retail_ii' / 'online_retail_II.xlsx',
}
COLUMNAS = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloque)
    return h.hexdigest()


def _nulo(v):
    return v is None or v is pd.NaT or (isinstance(v, float) and math.isnan(v))


def _texto(v):
    """Conserva el valor como texto; los nulos siguen siendo nulos; 489434.0 se guarda como '489434'."""
    if _nulo(v):
        return None
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)


def _numero(v, entero=False):
    if _nulo(v):
        return None
    return int(float(v)) if entero else float(v)


def _fecha_iso(v):
    if _nulo(v):
        return None
    return v if isinstance(v, str) else pd.Timestamp(v).strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
ruta = FUENTES[FUENTE]
if not ruta.exists():
    raise FileNotFoundError(f'No existe {ruta}. Para la fuente completa ejecuta antes:  python -m ia_mlops.dataset descargar')

if ruta.suffix.lower() == '.csv':
    hojas = [(ruta.name, pd.read_csv(ruta, dtype=object, keep_default_na=False, na_values=['']))]
else:
    hojas = list(pd.read_excel(ruta, sheet_name=None, dtype=object).items())

ejecutar_script('raw/01_crear_raw.sql')
total = 0
for nombre, df in hojas:
    filas = (
        (nombre, i + 2, _texto(r[0]), _texto(r[1]), _texto(r[2]), _numero(r[3], entero=True), _fecha_iso(r[4]),
         _numero(r[5]), _numero(r[6]), _texto(r[7]))
        for i, r in enumerate(df[COLUMNAS].itertuples(index=False, name=None))   # i + 2: la fila 1 es el encabezado
    )
    con.executemany('INSERT INTO raw_online_retail_ii (_hoja, _fila, Invoice, StockCode, Description, Quantity, InvoiceDate, '
                    'Price, Customer_ID, Country) VALUES (?,?,?,?,?,?,?,?,?,?)', filas)
    total += len(df)
    print(f'hoja {nombre!r}: {len(df):,} filas')
con.execute('INSERT INTO raw_carga (fuente, archivo, sha256, filas, cargado_en) VALUES (?,?,?,?,?)',
            (FUENTE, ruta.name, sha256_archivo(ruta), total, datetime.now().isoformat(timespec='seconds')))
con.commit()
print(f'RAW: {total:,} filas cargadas desde {ruta.name}')
pd.read_sql('SELECT * FROM raw_carga', con)

In [ ]:
pd.read_sql('SELECT * FROM raw_online_retail_ii LIMIT 8', con)

## 2. Mirar el dato crudo con ojo crítico
Antes de limpiar hay que **medir** qué está sucio. Cada hallazgo se convierte después en una regla de MASTER.

In [ ]:
hallazgos = {
    'filas': "SELECT COUNT(*) FROM raw_online_retail_ii",
    'facturas canceladas (empiezan por C)': "SELECT COUNT(*) FROM raw_online_retail_ii WHERE Invoice LIKE 'C%'",
    'sin Customer ID': "SELECT COUNT(*) FROM raw_online_retail_ii WHERE Customer_ID IS NULL",
    'cantidad <= 0': "SELECT COUNT(*) FROM raw_online_retail_ii WHERE Quantity <= 0",
    'precio <= 0': "SELECT COUNT(*) FROM raw_online_retail_ii WHERE Price <= 0",
    'codigos que no empiezan por 5 digitos': "SELECT COUNT(*) FROM raw_online_retail_ii WHERE UPPER(TRIM(StockCode)) NOT GLOB '[0-9][0-9][0-9][0-9][0-9]*'",
    'descripcion nula o vacia': "SELECT COUNT(*) FROM raw_online_retail_ii WHERE Description IS NULL OR TRIM(Description) = ''",
}
total_filas = escalar(hallazgos['filas'])
pd.DataFrame({'filas': {k: escalar(v) for k, v in hallazgos.items()}}).assign(
    porcentaje=lambda d: (100 * d['filas'] / total_filas).round(2))

In [ ]:
# Duplicados exactos: la misma fila repetida. Se conserva la primera y las demás se descartan con su motivo.
escalar("""
    SELECT COUNT(*) FROM (
      SELECT ROW_NUMBER() OVER (PARTITION BY Invoice, StockCode, Description, Quantity, InvoiceDate, Price,
                                IFNULL(Customer_ID, -1), Country ORDER BY _id) AS rn
      FROM raw_online_retail_ii) WHERE rn > 1""")

In [ ]:
# Códigos que NO son productos: envío, comisiones, ajustes, regalos, pruebas...
pd.read_sql("""
    SELECT UPPER(TRIM(StockCode)) AS codigo, MIN(Description) AS descripcion, COUNT(*) AS filas
    FROM raw_online_retail_ii
    WHERE UPPER(TRIM(StockCode)) NOT GLOB '[0-9][0-9][0-9][0-9][0-9]*'
    GROUP BY 1 ORDER BY filas DESC LIMIT 10""", con)

## 3. MASTER: depurar y modelar
`sql/master/01_crear_master.sql` crea las tablas (con claves y restricciones) y `02_poblar_master.sql` las llena desde RAW.
**Ábrelos y léelos: son SQL de verdad.** Reglas: recortar y pasar a mayúsculas, descartar duplicados con su motivo, separar en clientes, productos,
facturas y líneas, y marcar qué líneas son ventas válidas.

In [ ]:
ejecutar_script('master/01_crear_master.sql')
ejecutar_script('master/02_poblar_master.sql')
for tabla in ('master_clientes', 'master_productos', 'master_facturas', 'master_lineas', 'master_descartes'):
    print(f'{tabla:18s} {escalar(f"SELECT COUNT(*) FROM {tabla}"):>10,} filas')

## 4. Reglas de calidad
Cada regla es una consulta que devuelve un número. **Duras:** si fallan, el pipeline se detiene. **Alertas:** se registran y avisan.
**Informativas:** solo informan. El resultado queda en `master_calidad`.

In [ ]:
FECHA_MINIMA, FECHA_LIMITE = '2009-12-01', '2012-01-01'   # el límite es exclusivo
REGLAS = [  # (regla, descripción, consulta, tipo, umbral, cumple)
    ('R01_conciliacion', 'Filas de RAW menos (líneas de MASTER + descartes). Debe ser 0.',
     "SELECT (SELECT COUNT(*) FROM raw_online_retail_ii) - (SELECT COUNT(*) FROM master_lineas) - (SELECT COUNT(*) FROM master_descartes)", 'duro', '= 0', lambda v: v == 0),
    ('R02_facturas_sin_lineas', 'Facturas que no tienen ninguna línea.',
     "SELECT COUNT(*) FROM master_facturas f WHERE NOT EXISTS (SELECT 1 FROM master_lineas l WHERE l.factura_id = f.factura_id)", 'duro', '= 0', lambda v: v == 0),
    ('R03_integridad_referencial', 'Violaciones de claves foráneas.', None, 'duro', '= 0', lambda v: v == 0),
    ('R04_monto_coherente', 'Líneas donde monto_linea no es cantidad por precio.',
     "SELECT COUNT(*) FROM master_lineas WHERE ABS(monto_linea - ROUND(cantidad * precio_unitario, 2)) > 0.01", 'duro', '= 0', lambda v: v == 0),
    ('R05_fechas_en_rango', f'Facturas fuera del rango {FECHA_MINIMA} a {FECHA_LIMITE} (exclusivo).',
     f"SELECT COUNT(*) FROM master_facturas WHERE fecha_factura < '{FECHA_MINIMA}' OR fecha_factura >= '{FECHA_LIMITE}'", 'duro', '= 0', lambda v: v == 0),
    ('R06_facturas_con_varios_clientes', 'Facturas de RAW asociadas a más de un cliente.',
     "SELECT COUNT(*) FROM (SELECT TRIM(Invoice) FROM raw_online_retail_ii WHERE Customer_ID IS NOT NULL GROUP BY TRIM(Invoice) HAVING COUNT(DISTINCT Customer_ID) > 1)", 'alerta', '= 0', lambda v: v == 0),
    ('R07_cancelaciones_con_cantidad_positiva', 'Líneas de facturas canceladas con cantidad positiva.',
     "SELECT COUNT(*) FROM master_lineas l JOIN master_facturas f ON f.factura_id = l.factura_id WHERE f.es_cancelacion = 1 AND l.cantidad > 0", 'alerta', '= 0', lambda v: v == 0),
    ('R08_pct_lineas_sin_cliente', 'Porcentaje de líneas cuya factura no trae Customer ID (no entran en FS).',
     "SELECT ROUND(100.0 * SUM(CASE WHEN f.cliente_id IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) FROM master_lineas l JOIN master_facturas f ON f.factura_id = l.factura_id", 'info', 'informativo', None),
    ('R09_pct_ventas_validas', 'Porcentaje de líneas que son ventas válidas.',
     "SELECT ROUND(100.0 * SUM(es_venta_valida) / COUNT(*), 2) FROM master_lineas", 'info', 'informativo', None),
    ('R10_duplicados_descartados', 'Filas descartadas por ser duplicados exactos.',
     "SELECT COUNT(*) FROM master_descartes WHERE motivo = 'duplicado_exacto'", 'info', 'informativo', None),
    ('R11_lineas_precio_no_positivo', 'Líneas con precio menor o igual a cero.',
     "SELECT COUNT(*) FROM master_lineas WHERE precio_unitario <= 0", 'info', 'informativo', None),
    ('R12_codigos_que_no_son_producto', 'Códigos de StockCode que no son productos (envío, comisiones, ajustes...).',
     "SELECT COUNT(*) FROM master_productos WHERE es_producto = 0", 'info', 'informativo', None),
    ('R13_clientes_distintos', 'Clientes distintos en MASTER.', "SELECT COUNT(*) FROM master_clientes", 'info', 'informativo', None),
]

ahora = datetime.now().isoformat(timespec='seconds')
con.execute('DELETE FROM master_calidad')
errores = []
for regla, descripcion, consulta, tipo, umbral, cumple in REGLAS:
    valor = len(con.execute('PRAGMA foreign_key_check').fetchall()) if consulta is None else con.execute(consulta).fetchone()[0]
    estado = 'OK' if tipo == 'info' else ('OK' if cumple(valor) else ('ERROR' if tipo == 'duro' else 'ALERTA'))
    if estado == 'ERROR':
        errores.append(f'{regla}: {descripcion} -> {valor}')
    con.execute('INSERT INTO master_calidad (regla, descripcion, valor, umbral, estado, evaluado_en) VALUES (?,?,?,?,?,?)',
                (regla, descripcion, valor, umbral, estado, ahora))
con.commit()
assert not errores, 'Fallaron reglas de calidad duras:\n  ' + '\n  '.join(errores)
pd.read_sql('SELECT regla, valor, umbral, estado FROM master_calidad ORDER BY regla', con)

### Nada desaparece sin dejar rastro
Lo que no pasó a MASTER está en `master_descartes`, con su motivo. La regla R01 comprueba que `raw = master + descartes`.

In [ ]:
pd.read_sql("""
    SELECT 'RAW' AS capa, COUNT(*) AS filas FROM raw_online_retail_ii
    UNION ALL SELECT 'MASTER (líneas)', COUNT(*) FROM master_lineas
    UNION ALL SELECT 'descartes: ' || motivo, COUNT(*) FROM master_descartes GROUP BY motivo""", con)

In [ ]:
pd.read_sql("""
    SELECT f.factura_id, f.fecha_factura, f.cliente_id, l.codigo_producto, p.descripcion, l.cantidad, l.precio_unitario,
           l.monto_linea, l.es_venta_valida
    FROM master_lineas l
    JOIN master_facturas f ON f.factura_id = l.factura_id
    JOIN master_productos p ON p.codigo_producto = l.codigo_producto
    ORDER BY l.linea_id LIMIT 8""", con)

In [ ]:
con.close()
print('Listo: RAW y MASTER construidos en', DB)

## Para practicar (tarea de S1)
1. Añade una **regla de calidad** nueva a la lista `REGLAS` (por ejemplo, ventas con un monto desproporcionado) y confirma que aparece en `master_calidad`.
2. Cambia una regla de limpieza en `sql/master/02_poblar_master.sql` (por ejemplo, qué cuenta como producto) y vuelve a ejecutar. ¿Cambió la conciliación?
3. Explica en dos frases por qué RAW no se corrige.